# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arryan-56/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's (`content_hash_id`) aggregated organic-search and
engagement profile for a single calendar month. Built by grouping
`fact_content_daily_performance` by `content_hash_id` over `month=2026-03`,
joined to static attributes in `dim_content`.

I'm using `month=2026-03` (mid-panel) to build and test this contract.
`month=2026-06` (`_sample`) is the sealed final month — never touched for
logic, since it's the natural past→future outcome window.

In [10]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
rel = "hf://datasets/FlyRank/internship-warehouse"
fact_path = f"{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet"
dim_path = f"{rel}/dim_content.parquet"

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_key_combos,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{fact_path}')
""")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬────────────────────────┬────────────┬────────────┐
│ total_rows │ distinct_key_combos │ distinct_content_items │  min_date  │  max_date  │
│   int64    │        int64        │         int64          │    date    │    date    │
├────────────┼─────────────────────┼────────────────────────┼────────────┼────────────┤
│    9841378 │             9841378 │                 331437 │ 2026-03-01 │ 2026-03-31 │
└────────────┴─────────────────────┴────────────────────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (built into the frame, knowable at decision moment):
- `avg_ctr` = gsc_clicks / gsc_impressions
- `avg_position` = gsc_avg_position
- `engagement_rate` = ga4_engaged_sessions / ga4_sessions
- `avg_session_duration_sec` = ga4_total_engagement_sec / ga4_sessions
- `content_age_days` = report date − content_created_date

**Label/proxy:** none supervised — this is unsupervised. The proxy target is a
performance-archetype cluster assignment derived from the features above,
used to group content for action (e.g. high-visibility engaged /
high-traffic low-engagement / low-visibility / declining).

**Context:** `content_type`, `main_intent`, `search_volume`, `competition_level`
— used to describe/interpret clusters after they form, not as clustering inputs.

**Excluded:**
- `fact_content_query_90d` (whole table) — query-level grain, too granular for
  a content-item archetype; would explode row count without adding signal here.
- AI-referral columns (`ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`,
  `ai_claude`, `ai_meta`, `ai_other`) and channel-split sessions
  (`sessions_direct`, `sessions_social`, `sessions_paid`, `sessions_referral`)
  — real signals, but sparse/noisy for a first-pass backbone. Candidate for a
  later refinement, not this contract.

In [13]:
preview = con.sql(f"""
    WITH monthly_agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_clicks)               AS total_clicks,
            SUM(gsc_impressions)          AS total_impressions,
            AVG(gsc_avg_position)         AS avg_position,
            SUM(ga4_sessions)             AS total_sessions,
            SUM(ga4_engaged_sessions)     AS total_engaged_sessions,
            SUM(ga4_total_engagement_sec) AS total_engagement_sec,
            MAX(report_date)              AS last_report_date
        FROM read_parquet('{fact_path}')
        GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        m.total_clicks / NULLIF(m.total_impressions, 0)            AS avg_ctr,
        m.avg_position,
        m.total_engaged_sessions / NULLIF(m.total_sessions, 0)     AS engagement_rate,
        m.total_engagement_sec / NULLIF(m.total_sessions, 0)       AS avg_session_duration_sec,
        DATE_DIFF('day', d.content_created_date, m.last_report_date) AS content_age_days,
        d.content_type, d.main_intent, d.search_volume, d.competition_level
    FROM monthly_agg m
    JOIN read_parquet('{dim_path}') d ON m.content_hash_id = d.content_hash_id
""")
print(preview.limit(5))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬───────────────────────┬────────────────────┬─────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────┬───────────────┬───────────────────┐
│     content_hash_id      │        avg_ctr        │    avg_position    │ engagement_rate │ avg_session_duration_sec │ content_age_days │  content_type   │  main_intent  │ search_volume │ competition_level │
│         varchar          │        double         │       double       │     double      │          double          │      int64       │     varchar     │    varchar    │     int64     │      varchar      │
├──────────────────────────┼───────────────────────┼────────────────────┼─────────────────┼──────────────────────────┼──────────────────┼─────────────────┼───────────────┼───────────────┼───────────────────┤
│ content_7a105f548d9c6916 │ 0.0010731258623332823 │    7.2095494029688 │             0.0 │                      0.0 │              396 │ keyword article │ informationa

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three claims verified below: (a) row count + distinct content items for the
March slice, (b) data availability using `IS TRUE`, (c) missing-value check on
the core metric columns feeding the five features.

In [14]:
counts_and_window = con.sql(f"""
    SELECT COUNT(*) AS daily_row_count,
           COUNT(DISTINCT content_hash_id) AS distinct_content_items,
           MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{fact_path}')
""")
print(counts_and_window)

┌─────────────────┬────────────────────────┬────────────┬────────────┐
│ daily_row_count │ distinct_content_items │  min_date  │  max_date  │
│      int64      │         int64          │    date    │    date    │
├─────────────────┼────────────────────────┼────────────┼────────────┤
│         9841378 │                 331437 │ 2026-03-01 │ 2026-03-31 │
└─────────────────┴────────────────────────┴────────────┴────────────┘



Code cell (availability, IS TRUE):

In [15]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS both_available_rows
    FROM read_parquet('{fact_path}')
""")
print(availability)

┌────────────┬────────────────────┬────────────────────┬─────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │ both_available_rows │
│   int64    │       int64        │       int64        │        int64        │
├────────────┼────────────────────┼────────────────────┼─────────────────────┤
│    9841378 │            3611061 │             413966 │              364347 │
└────────────┴────────────────────┴────────────────────┴─────────────────────┘



Code cell (missing values on the metric columns behind your five features):

In [16]:
missingness = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(gsc_clicks) AS null_gsc_clicks,
        COUNT(*) - COUNT(gsc_impressions) AS null_gsc_impressions,
        COUNT(*) - COUNT(gsc_avg_position) AS null_gsc_avg_position,
        COUNT(*) - COUNT(ga4_sessions) AS null_ga4_sessions,
        COUNT(*) - COUNT(ga4_engaged_sessions) AS null_ga4_engaged_sessions
    FROM read_parquet('{fact_path}')
""")
print(missingness)

┌────────────┬─────────────────┬──────────────────────┬───────────────────────┬───────────────────┬───────────────────────────┐
│ total_rows │ null_gsc_clicks │ null_gsc_impressions │ null_gsc_avg_position │ null_ga4_sessions │ null_ga4_engaged_sessions │
│   int64    │      int64      │        int64         │         int64         │       int64       │           int64           │
├────────────┼─────────────────┼──────────────────────┼───────────────────────┼───────────────────┼───────────────────────────┤
│    9841378 │               0 │                    0 │               6230317 │           3018741 │                   3018741 │
└────────────┴─────────────────┴──────────────────────┴───────────────────────┴───────────────────┴───────────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- Unbalanced panel: clients have different `gsc_data_start` / `ga4_data_start`
  (per `dim_clients`), so March 2026 doesn't have equal history depth across
  every client — a content item under a newer client has a thinner, noisier
  monthly aggregate than one under a longer-tenured client.
- `gsc_data_available` / `ga4_data_available` can be false for some rows even
  within `month=2026-03` — this data can't tell you performance for a content
  item during periods where the source system simply wasn't connected yet.
- Content-level aggregation hides day-to-day volatility within the month —
  this contract can't tell you *when* in March a spike or drop happened, only
  the monthly total/average.

In [17]:
history_depth = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM read_parquet('{rel}/dim_clients.parquet')
    ORDER BY gsc_data_start
    LIMIT 10
""")
print(history_depth)

┌─────────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ gsc_data_start │ ga4_data_start │
│         varchar         │      date      │      date      │
├─────────────────────────┼────────────────┼────────────────┤
│ client_9958f0a7ae1df715 │ 2025-01-27     │ 2025-10-29     │
│ client_ff644d8251367cbb │ 2025-01-27     │ 2025-10-29     │
│ client_73cda7b4e4f265ea │ 2025-02-11     │ 2026-03-24     │
│ client_fef1a8f436438636 │ 2025-03-11     │ 2026-03-06     │
│ client_62f4a7e64f5e0096 │ 2025-06-07     │ NULL           │
│ client_b10cb2997d0c7c86 │ 2025-06-18     │ 2025-11-15     │
│ client_65de48885f4ef01b │ 2025-06-21     │ 2026-02-19     │
│ client_c182d11e4862a37d │ 2025-06-21     │ 2026-02-20     │
│ client_3197e6291363b4db │ 2025-06-29     │ 2025-11-09     │
│ client_625b6439094e23e4 │ 2025-07-01     │ 2026-02-19     │
├─────────────────────────┴────────────────┴────────────────┤
│ 10 rows                                         3 columns │
└───────

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.